# Amparo -- M3 · S10: RAG agentico

Sobre el retrieval avanzado de S08 (hybrid + rerank, configuracion C) se montan
las dos rutas agenticas de la S10 y se corren sobre el mismo eval set que el RAG
de una pasada, para decidir con datos si un agente se justifica:

| Ruta | Codigo | Que decide el modelo |
|---|---|---|
| **Una pasada** (base, config C) | `pipeline.answer_query` | nada: recupera siempre, una vez |
| **Tool use** (Lab A) | `tools.responder_con_tools` | si busca y con que consulta |
| **ReAct** (Lab B) | `agentico.agente_react` | que acciones encadena: buscar, calcular, responder |

Toda la logica vive en `tools/rag/` (con tests que corren sin GPU); aca se clona
el repo, se instala el stack pesado y se corren las fases.

**Requiere GPU** (T4 o superior) y el indice FAISS construido por
`m3_s07_rag_ingenuo.ipynb` en Drive.

Las tres rutas devuelven el mismo contrato (`pipeline.to_eval_record`), con los
campos `sistema` y `traza`: las corridas quedan listas para evaluarse con RAGAS y
el harness.

In [ ]:
# Rama del repo a clonar. Cambiala si el codigo del RAG avanzado todavia no esta
# en main (por ejemplo, mientras vive en una rama de PR).
RAMA = "RAGAS"

import os

if not os.path.isdir("Amparo"):
    !git clone -b {RAMA} https://github.com/TomasPosada0626/Amparo.git
else:
    !cd Amparo && git fetch origin && git checkout {RAMA} && git pull

%cd Amparo

In [ ]:
import os, sys
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

REQUERIDOS = [
    "tools/rag/hybrid.py",
    "tools/rag/rerank.py",
    "tools/rag/tools.py",
    "tools/rag/agentico.py",
    "tools/rag/pipeline.py",
    "data/eval_set.json",
]
faltan = [r for r in REQUERIDOS if not os.path.exists(r)]
if faltan:
    raise SystemExit(
        f"Falta en la rama '{RAMA}': {faltan}\n\n"
        "Este notebook corre contra el repo REMOTO. Si el codigo del RAG agentico "
        "todavia esta solo en tu maquina, commitealo y pusheralo, o cambia RAMA."
    )
print("Repo OK.")

In [ ]:
# Dependencias livianas del repo (incluye faiss-cpu y rank_bm25, el BM25 de la
# hybrid search).
!pip install -q -r requirements.txt

In [ ]:
# Stack pesado: se instala aqui y no en requirements.txt, para no reemplazar el
# build de PyTorch con CUDA que Colab ya trae. sentence-transformers trae el
# cross-encoder del reranking; transformers alcanza para e5 y la generacion;
# peft/bitsandbytes solo si se genera con el adaptador LoRA.
!pip install -q -U transformers sentence-transformers peft bitsandbytes accelerate

## Fase 1 -- Cargar el indice, el BM25 y el generador

Mismo punto de partida que `m3_s08_rag_avanzado.ipynb`: el indice denso se copia
de Drive, el BM25 se construye una vez en memoria y Qwen2.5-7B se carga una sola
vez y se reusa en todas las rutas.

In [ ]:
from google.colab import drive
import pathlib, shutil
from tools.rag import config

drive.mount("/content/drive")
origen = pathlib.Path(config.DRIVE_ROOT) / "rag"
config.ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
for nombre in ("rag_index.faiss", "rag_index_metadata.jsonl"):
    shutil.copy(origen / nombre, config.ARTIFACTS_DIR / nombre)
print("Indice copiado de Drive a artifacts/.")

In [ ]:
from tools.rag import pipeline
from tools.rag.hybrid import BM25Index

store = pipeline.load_index()
bm25 = BM25Index(store.metadata)   # una sola vez; se reusa en cada consulta
print(f"Indice denso: {len(store)} chunks | BM25: {len(bm25)} chunks")

In [ ]:
USE_LORA = False
model_bundle = pipeline.load_model(use_lora=USE_LORA)
print("Modelo cargado.")

## Fase 2 -- Lab A · Tool use: el retrieval como herramienta

En vez de recuperar siempre, el modelo decide si llamar `buscar_normas` y con
que consulta (puede reformular la pregunta coloquial). La herramienta invoca el
retrieval avanzado (configuracion C). Dos casos: uno que amerita buscar
(fundamentar en una norma) y uno que no (un saludo).

In [ ]:
from tools.rag import tools

for consulta in [
    "me despidieron sin justa causa, que derechos tengo",   # deberia buscar
    "hola, buenas tardes",                                    # no deberia buscar
]:
    print("=" * 78)
    print("USUARIO:", consulta)
    r = tools.responder_con_tools(consulta, store, bm25=bm25, model_bundle=model_bundle, use_lora=USE_LORA)
    for tc in r["tool_calls"]:
        print(f"  -> llamo {tc['tool']}({tc['args']})")
    print(f"  contexts recuperados: {r['n_retrieved']}")
    print("AMPARO:", r["response"])

## Fase 3 -- Lab B · Mini-agente ReAct

El agente escribe un pensamiento, elige una accion (`buscar_normas`,
`calculadora`, `calcular_plazo` o `Responder`), lee la observacion y repite.
Tres consultas:

1. **Simple**: una sola norma alcanza. El agente deberia buscar y responder; si da
   mas vueltas, solo suma latencia.
2. **Compuesta (arriendo)**: necesita la norma *y* una cuenta: el tope del
   reajuste del canon (Ley 820 de 2003, articulo 20: hasta el 100 % del IPC)
   aplicado a los datos del usuario.
3. **Plazo (derecho de peticion)**: buscar el termino para responder, contar los
   dias habiles con festivos y, si no responden, buscar que hacer (tutela).

Antes de aceptar la respuesta, el codigo verifica que cada articulo citado haya
aparecido en una observacion; si no, la rechaza y el agente corrige (se ve en la
traza como `Responder (rechazado)`).

In [ ]:
from tools.rag import agentico

consultas_react = {
    "SIMPLE": "cuanto tiempo tiene una entidad para responder un derecho de peticion",
    "COMPUESTA": ("pago 1.200.000 de arriendo y el IPC del año pasado fue 5,2 %, "
                  "hasta cuanto me pueden subir el canon"),
    "PLAZO": ("radique un derecho de peticion el 1 de septiembre de 2026, cuando me "
              "deben responder y si no me responden que puedo hacer"),
}

for etiqueta, consulta in consultas_react.items():
    print("=" * 78)
    print(f"{etiqueta}: {consulta}")
    r = agentico.agente_react(consulta, store, bm25=bm25, model_bundle=model_bundle, use_lora=USE_LORA)
    for p in r["traza"]:
        print(f"  [{p['paso']}] {p['accion']}[{p['argumento'][:70]}]")
        if p["observacion"]:
            print(f"       obs: {p['observacion'][:100]!r}")
    print("AMPARO:", r["response"])

## Fase 4 -- Corrida de las tres rutas sobre el eval set (con latencia)

Mismo eval set (`data/eval_set.json`: gold + adversariales), mismo generador y
mismo retrieval (configuracion C) en las tres rutas: lo unico que cambia es
quien decide cuando y cuanto recuperar. Se mide la latencia por consulta, porque
cada paso del agente es una generacion mas y ese costo se reporta junto a la
calidad.

La ruta de una pasada se vuelve a correr aca (y no se reusa la corrida C de S08)
para que las tres salgan de la misma sesion y del mismo modelo cargado.

In [ ]:
import json, time
from tools.evaluation import eval_set as eval_set_mod

registros = eval_set_mod.load_eval_set()
print(f"{len(registros)} registros ({len(eval_set_mod.gold_examples(registros))} gold, "
      f"{len(eval_set_mod.adversarial_examples(registros))} adversariales)")

RUTAS = {
    "una_pasada": lambda q: pipeline.answer_query(
        q, store, use_lora=USE_LORA, bm25=bm25, model_bundle=model_bundle,
        use_hybrid=True, use_rerank=True),
    "tool_use": lambda q: tools.responder_con_tools(
        q, store, bm25=bm25, model_bundle=model_bundle, use_lora=USE_LORA),
    "react": lambda q: agentico.agente_react(
        q, store, bm25=bm25, model_bundle=model_bundle, use_lora=USE_LORA),
}

corridas, latencias = {}, {}
for ruta, responder in RUTAS.items():
    print(f"\nCorriendo ruta {ruta} ...")
    records, t0 = [], time.perf_counter()
    for i, reg in enumerate(registros, start=1):
        resultado = responder(reg["messages"][1]["content"])
        records.append(pipeline.to_eval_record(resultado, reg))
        if i % 10 == 0 or i == len(registros):
            print(f"  {i}/{len(registros)} -- {(time.perf_counter() - t0) / i:.1f} s/consulta")
    latencias[ruta] = (time.perf_counter() - t0) / len(registros)
    corridas[ruta] = records

In [ ]:
# Persistir en Drive para evaluar (RAGAS + harness) sin re-generar.
destino = pathlib.Path(config.DRIVE_ROOT) / "rag" / "corridas_s10"
destino.mkdir(parents=True, exist_ok=True)
etiqueta = "lora" if USE_LORA else "base"

for ruta, records in corridas.items():
    ruta_archivo = destino / f"eval_records_{ruta}_{etiqueta}.json"
    with open(ruta_archivo, "w", encoding="utf-8") as f:
        json.dump(records, f, ensure_ascii=False, indent=2)
    print(f"  {ruta} -> {ruta_archivo}")

with open(destino / f"latencias_{etiqueta}.json", "w", encoding="utf-8") as f:
    json.dump(latencias, f, indent=2)

print("\nResumen por ruta:")
for ruta, records in corridas.items():
    sin_ctx = sum(1 for r in records if r["n_retrieved"] == 0)
    pasos = sum(len(r["traza"]) for r in records) / len(records)
    print(f"  {ruta:<11} {latencias[ruta]:6.1f} s/consulta | sin contexto: {sin_ctx:>2} | "
          f"pasos promedio: {pasos:.1f}")

---
### Contenido

1. **Tool use** (`tools/rag/tools.py`) y **ReAct** (`tools/rag/agentico.py`), con
   tests que corren sin GPU (`tests/rag/test_tools.py`, `tests/rag/test_agentico.py`).
2. Las corridas de las tres rutas en formato RAGAS, con su traza y su latencia
   (Fase 4), en `corridas_s10/` de Drive.

*Amparo · M3 · S10 · RAG agentico.*